In [4]:
import cv2
import numpy as np
import mediapipe as mp
import threading
from scipy.signal import find_peaks, butter, filtfilt
import time
from collections import deque
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# MediaPipe 초기화
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

frame_lock = threading.Lock()  # 스레드간 동기화를 위한 락
current_frame = None  # 현재 프레임 저장
is_frame_available = False  # 프레임이 있는지 여부

def read_frame(cap):
    global current_frame, is_frame_available
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 프레임을 좌우 반전
        frame = cv2.flip(frame, 1)

        # 락을 이용해 현재 프레임 업데이트
        with frame_lock:
            current_frame = frame
            is_frame_available = True


def display_triangular_regions(frame, regions):
    """
    화면에 삼각형 영역을 표시하는 함수.
    """
    for triangle_points in regions:
        # 삼각형을 그리기 위한 다각형 선 그리기
        cv2.polylines(frame, [triangle_points], isClosed=True, color=(0, 255, 0), thickness=2)

# ROI에서 이마, 왼쪽 볼 위/아래, 오른쪽 볼 위/아래, 턱의 6개 구간으로 RGB 신호 추출

def extract_rgb_signals_by_face_regions(frame, landmarks):
    h, w, _ = frame.shape

    # 필요한 랜드마크 인덱스
    forehead_indices = [36, 205, 206]  # 왼쪽 볼 안쪽
    left_cheek_upper_indices = [117, 100, 50]  # 왼쪽 볼 위
    left_cheek_lower_indices = [187, 50, 205]  # 왼쪽 볼 아래
    right_cheek_upper_indices = [346, 329, 280]  # 오른쪽 볼 위
    right_cheek_lower_indices = [280, 425, 411]  # 오른쪽 볼 아래
    chin_indices = [266, 425, 426]  # 왼쪽 볼 안쪽
    
    # ROI 영역 설정 (삼각형 꼭짓점 설정)
    roi_landmarks = [
        forehead_indices, left_cheek_upper_indices, left_cheek_lower_indices,
        right_cheek_upper_indices, right_cheek_lower_indices, chin_indices
    ]

    mean_rgbs = []
    regions = []
    for indices in roi_landmarks:
        # 삼각형 꼭짓점 좌표 생성
        triangle_points = np.array([[int(landmarks[idx].x * w), int(landmarks[idx].y * h)] for idx in indices])

        # 빈 마스크 생성
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [triangle_points], 255)  # 삼각형 영역을 하얗게 채우기

        # 원본 이미지에서 삼각형 영역 추출
        masked_roi = cv2.bitwise_and(frame, frame, mask=mask)

        # 삼각형 영역의 RGB 평균 값 계산
        mean_rgb = cv2.mean(masked_roi, mask=mask)[:3]  # 알파 채널 제외
        mean_rgbs.append(mean_rgb)

        # 삼각형 영역의 좌표를 그대로 저장
        regions.append(triangle_points)

    return np.array(mean_rgbs), regions

def pca_algorithm(S, n_components=1):
    """
    S: N x 3 행렬 (RGB 신호의 시간에 따른 변화)
    n_components: 추출할 주성분의 개수
    """
    S = np.array(S)
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(S)
    
    # 첫 번째 주성분 반환 (심박수와 관련된 신호로 가정)
    return principal_components[:, 0]

# 수정된 POS 알고리즘 적용
def pos_algorithm(S):
    S = np.array(S)
    mean_color = np.mean(S, axis=0)
    C = (S / mean_color) - 1
    P = np.array([0, 1, -1])
    return np.dot(C, P)

# 대역 통과 필터 적용
def bandpass_filter(signal, lowcut=0.7, highcut=4.0, fs=60.0, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

# 심박수 계산 함수
def calculate_heart_rate(signal, fps):
    peaks, _ = find_peaks(signal, distance=fps * 0.5)
    peak_intervals = np.diff(peaks) / fps
    return 60.0 / np.mean(peak_intervals) if len(peak_intervals) > 0 else 0

def moving_average(data, window_size=5):
    data = list(data)
    return np.mean(data) if len(data) < window_size else np.mean(data[-window_size:])

# Matplotlib Figure를 NumPy 배열로 변환하는 함수
"""
def fig2data(fig):
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return buf
"""
def fig2data(fig):
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))  # RGBA 형식이므로 마지막 차원이 4
    buf = buf[:, :, :3]  # RGB 값만 추출
    return buf

# FFT를 이용해 심박수 계산 함수 추가
def calculate_heart_rate_fft(signal, sampling_rate):
    N = len(signal)
    fft_result = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1.0/sampling_rate)

    fft_magnitude = np.abs(fft_result)
    positive_freqs = freqs[:N // 2]
    positive_magnitude = fft_magnitude[:N // 2]

    # 심박수 범위: 0.75Hz ~ 3.0Hz (45 bpm ~ 180 bpm)
    min_bpm = 45
    max_bpm = 180
    min_freq = min_bpm / 60.0
    max_freq = max_bpm / 60.0

    valid_indices = np.where((positive_freqs >= min_freq) & (positive_freqs <= max_freq))
    valid_freqs = positive_freqs[valid_indices]
    valid_magnitude = positive_magnitude[valid_indices]

    if len(valid_freqs) > 0:
        peak_freq_index = np.argmax(valid_magnitude)
        peak_freq = valid_freqs[peak_freq_index]
        heart_rate_bpm = peak_freq * 60.0
        return heart_rate_bpm
    else:
        return 0

def weighted_combination_signal(pos_signal_history, weights):
    # pos_signal_history는 각 ROI의 시계열 신호들이 저장된 배열 리스트입니다 (6개의 ROI)
    # weights는 각 ROI에 대한 가중치 벡터입니다
    pos_signals = np.array(pos_signal_history)  # 6 x T 형태 (T는 시간축 길이)
    
    # 가중치 벡터 적용하여 결합된 신호 생성
    combined_signal = np.dot(weights, pos_signals)
    return combined_signal
"""
def main():
    #video_path = "Videos/KakaoTalk_20241113_222046672.mp4"
    video_path = "OneDrive/바탕 화면/종설 인터뷰영상/s1/vid_s1_T1.avi"
    cap = cv2.VideoCapture(video_path)
    #cap = cv2.VideoCapture(0)
    #fps = cap.get(cv2.CAP_PROP_FPS) or 30
    fps = 30
"""
def process_frame():
    global current_frame, is_frame_available
    rgb_signals_history = [deque(maxlen=300) for _ in range(6)]
    heart_rate_history = deque(maxlen=5)
    last_heart_rate = None
    last_display_time = time.time()
    update_interval = 4


    bpm_values = deque(maxlen=10 * 30)  # 약 10초 동안의 데이터 (30 FPS 기준)
    #bpm_values = deque(maxlen=20)
    avg_bpm = 0
    
    #플롯 수정정
    plt.switch_backend('agg')  # 백엔드 설정
    fig, axes = plt.subplots(3,1,figsize=(6, 6))  # 그래프 크기 확대
    lines=[]
    for ax in axes:
        line, = ax.plot([], [])
        lines.append(line)
        ax.set_xlim(0, 300)  # 최대 데이터 길이
    plt.close()  # plt.show()로 표시되지 않도록 닫음

    pos_signal_history = [deque(maxlen=300) for _ in range(6)]
    amplification_factor = 50
    #평균시그널 초기화
    aver_signal_history=deque(maxlen=300)
    #내적시그널 초기화
    combined_signal=deque(maxlen=300)
    #가중치설정
    weights = np.array([1, 2, 1, 2, 1, 1]) / 6

    #프레임 초기화
    frame_count = 0
    
    while True:
        start_time = time.time() 
        with frame_lock:
            if not is_frame_available:
                continue
            frame = current_frame.copy()
            
        # MediaPipe 얼굴 랜드마크 탐지
        results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            # RGB 신호 추출
            rgb_signals, regions = extract_rgb_signals_by_face_regions(frame, face_landmarks)

            # 구간별로 심박수 계산
            heart_rates = []
            for i in range(6):
                rgb_signals_history[i].append(rgb_signals[i])
                if len(rgb_signals_history[i]) > int(fps * 5):
                    
                    pos_signal = pos_algorithm(rgb_signals_history[i])
                    filtered_signal = bandpass_filter(pos_signal, fs=fps)
                    
                    """
                    #PCA알고리즘을 이용한 시그널
                    pca_signal = pca_algorithm(rgb_signals_history[i])
                    filtered_signal = bandpass_filter(pca_signal, fs=fps)
                    """
                    #FFT를 이용한 HR계산
                    #heart_rate = calculate_heart_rate_fft(filtered_signal, fps)
                    #heart_rate = calculate_heart_rate(filtered_signal, fps)
                    #heart_rates.append(heart_rate)
                    
                    standardized_signal = (filtered_signal - np.mean(filtered_signal)) / np.std(filtered_signal)
                    amplified_signal = standardized_signal[-1] * amplification_factor  # 신호 증폭
                    pos_signal_history[i].append(amplified_signal)
                    
                    """
                    if i == 0:  # 첫 번째 구간의 pos_signal을 그래프로 표시
                        # 신호 표준화
                        
                        # 그래프 업데이트
                        ax=axes[0]
                        xdata0 = np.arange(len(pos_signal_history[i]))
                        ydata0 = np.array(pos_signal_history[i])
                        lines[0].set_data(xdata0, ydata0)
                        ax.set_xlim(0, max(100, len(pos_signal_history[i])))
                        # y축 범위를 신호의 최소/최대값 기반으로 자동 조정
                        ax.set_ylim(np.min(ydata0) - 1, np.max(ydata0) + 1)

                        # Figure를 이미지로 변환
                        plot_img0 = fig2data(fig)
                        plot_img0 = cv2.cvtColor(plot_img0, cv2.COLOR_RGB2BGR)
                        cv2.imshow('POS Signal', plot_img0)   
                    """
                    """
                    if i == 1:  # 첫 번째 구간의 pos_signal을 그래프로 표시
                        # 신호 표준화
                        #pos_signal_history[1].append(amplified_signal)
                        # 그래프 업데이트
                        ax=axes[1]
                        xdata1 = np.arange(len(pos_signal_history[i]))
                        ydata1 = np.array(pos_signal_history[i])
                        lines[1].set_data(xdata1, ydata1)
                        ax.set_xlim(0, max(100, len(pos_signal_history[i])))
                        # y축 범위를 신호의 최소/최대값 기반으로 자동 조정
                        ax.set_ylim(np.min(ydata1) - 1, np.max(ydata1) + 1)

                        # Figure를 이미지로 변환
                        plot_img1 = fig2data(fig)
                        plot_img1 = cv2.cvtColor(plot_img1, cv2.COLOR_RGB2BGR)
                        cv2.imshow('POS Signal', plot_img1)   
                        
                    if i == 3:  # 첫 번째 구간의 pos_signal을 그래프로 표시
                        # 신호 표준화
                        #pos_signal_history[3].append(amplified_signal)
                        # 그래프 업데이트
                        ax=axes[2]
                        xdata3 = np.arange(len(pos_signal_history[i]))
                        ydata3 = np.array(pos_signal_history[i])
                        lines[2].set_data(xdata3, ydata3)
                        ax.set_xlim(0, max(100, len(pos_signal_history[i])))
                        # y축 범위를 신호의 최소/최대값 기반으로 자동 조정
                        ax.set_ylim(np.min(ydata3) - 1, np.max(ydata3) + 1)

                        # Figure를 이미지로 변환
                        plot_img3 = fig2data(fig)
                        plot_img3 = cv2.cvtColor(plot_img3, cv2.COLOR_RGB2BGR)
                        cv2.imshow('POS Signal', plot_img3)
                    """

            """
            #평균 시그널 그래프
            aver_signal_history=np.mean(pos_signal_history, axis=0)
            """
            
            if len(pos_signal_history[0]) > 0:
                #내적을 이용한 시그널그프프
                combined_signal = weighted_combination_signal(pos_signal_history, weights)
                ax=axes[0]
                xdata0 = np.arange(len(combined_signal))
                ydata0 = np.array(combined_signal)
                lines[0].set_data(xdata0, ydata0)
                ax.set_xlim(0, max(100, len(combined_signal)))
                # y축 범위를 신호의 최소/최대값 기반으로 자동 조정
                ax.set_ylim(np.min(ydata0) - 1, np.max(ydata0) + 1)
    
                # Figure를 이미지로 변환
                plot_img0 = fig2data(fig)
                plot_img0 = cv2.cvtColor(plot_img0, cv2.COLOR_RGB2BGR)
                cv2.imshow('POS Signal', plot_img0)

            #FFT를 이용한 HR계산
            #heart_rate = calculate_heart_rate_fft(aver_signal_history, fps)
            heart_rate = calculate_heart_rate(combined_signal, fps)
            heart_rates.append(heart_rate)
            """                
            if len(heart_rates) >= 3:
                median_hr = np.median(heart_rates)
                differences = np.abs(heart_rates - median_hr)
                top_indices = np.argsort(differences)[:3]
                top_3_heart_rates = np.array(heart_rates)[top_indices]
                final_heart_rate = np.mean(top_3_heart_rates)
                heart_rate_history.append(final_heart_rate)
                smoothed_heart_rate = moving_average(heart_rate_history)
            """
            if len(heart_rates) >0:
                smoothed_heart_rate = moving_average(heart_rates)
                current_time = time.time()
                if current_time - last_display_time > update_interval:
                    if smoothed_heart_rate is not None:
                        last_heart_rate = smoothed_heart_rate
                    last_display_time = current_time

                # 실시간 BPM 추가 및 10초 평균 계산
                if last_heart_rate is not None and last_heart_rate > 0:
                    bpm_values.append(last_heart_rate)
                if len(bpm_values) > 5:
                    avg_bpm = np.mean(bpm_values)  # 10초 평균 BPM

                display_triangular_regions(frame, regions)


            if last_heart_rate is not None:
                cv2.putText(frame, f'Heart Rate: {int(last_heart_rate)} BPM', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            if avg_bpm > 0:
                cv2.putText(frame, f'10-sec Avg BPM: {int(avg_bpm)}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow('rPPG Heart Rate Monitor with 6 ROIs', frame)
        # 원래 FPS에 맞춰 대기 시간 계산
        elapsed_time = time.time() - start_time
        wait_time = max(1.0 / fps - elapsed_time, 0)
        time.sleep(wait_time)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()

if __name__ == "__main__":
    # 비디오 캡처 초기화
    video_path = "OneDrive/바탕 화면/종설 인터뷰영상/s1/vid_s1_T1.avi"
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30

    # 스레드 생성
    read_thread = threading.Thread(target=read_frame, args=(cap,))
    process_thread = threading.Thread(target=process_frame)

    # 스레드 시작
    read_thread.start()
    process_thread.start()

    # 스레드 종료 대기
    read_thread.join()
    process_thread.join()

    # 비디오 캡처 종료
    cap.release()
